# Hybrid-vs-FT judgment run — LoRA holdout inference + v5 fold decomposition

One GPU session, two sequential stages (models never co-resident). The full-FT
checkpoint lost weight shards 1-4 on Drive, so the FT arm is the **Qwen3-VL-8B
LoRA adapter** (`qwen3vl_final`); its clean holdout is the LoRA notebook's own
fold 0 (n=1,915, sklearn-1.6.1 fold assignment reproduced locally).

1. **Stage B** — base Qwen3-VL-8B + LoRA adapter inference on the 1,915-sample
   fold-0 holdout. No training-time EM was recorded for this adapter — this is
   the first clean measurement.
2. **Stage C1** — caption-only v5 event decomposition for the same 1,915 Ids
   (193 reused from the 900-sample run's state, ~1,722 newly generated).

Inputs on Drive: `snu ai/snu_colab_full_train_bundle.tar` (existing),
`snu ai/snu_colab_hybrid_judgment_bundle.tar` (code bundle, uploaded fresh),
`snu ai/snu_qwen3vl_outputs/qwen3vl_final/` (LoRA adapter, existing).

Outputs on Drive (both resumable — rerunning a stage skips completed rows):
`snu ai/outputs/lora_holdout_infer/` and
`snu ai/preprocessing/qwen3vl_event_chunks_v5_text_holdout/`.


In [1]:
import os
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ['HF_HUB_DISABLE_XET'] = '1'

DRIVE_ROOT = Path('/content/drive/MyDrive/snu ai')
CODE_BUNDLE_PATH = DRIVE_ROOT / 'snu_colab_hybrid_judgment_bundle.tar'
EXPECTED_BUNDLE_SHA256 = 'D73299C7C952020D21CFAC0FA64D0E74946B1FF1825340B6873384C8811268B3'
WORKDIR = Path('/content/snu_ai')
DATA_ROOT = WORKDIR / 'snuaichallenge_data'
TRAIN_CSV = DATA_ROOT / 'train.csv'
TRAIN_IMAGE_DIR = DATA_ROOT / 'train'
HOLDOUT_IDS = WORKDIR / 'outputs/lora_fold_split/holdout_ids.csv'
SEED_STATE = WORKDIR / 'preprocessing/generated/qwen3vl_event_chunks_v5_text_900/state.jsonl'

BASE_MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'
ADAPTER_DIR = DRIVE_ROOT / 'snu_qwen3vl_outputs/qwen3vl_final'
OUT_FT = DRIVE_ROOT / 'outputs/lora_holdout_infer'
OUT_V5 = DRIVE_ROOT / 'preprocessing/qwen3vl_event_chunks_v5_text_holdout'

HF_HOME = Path('/content/huggingface')
os.environ['HF_HOME'] = str(HF_HOME)

MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'
MODEL_REVISION = '0c351dd01ed87e9c1b53cbc748cba10e6187ff3b'
FT_BATCH_SIZE = 8
V5_QUANTIZATION = 'auto'
V5_BATCH_SIZE = 0
V5_MAX_NEW_TOKENS = 384
V5_MAX_RETRIES = 1
HOLDOUT_EXPECTED = 1915
V5_SEED_OVERLAP_EXPECTED = 193


## 1. Mount Drive, install dependencies, extract the two bundles

In [2]:
import hashlib
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    'transformers==5.13.0', 'accelerate==1.14.0', 'bitsandbytes>=0.46',
    'qwen-vl-utils==0.0.14', 'peft', 'torchao>=0.16.0',
], check=True)


def find_drive_file(filename):
    direct = DRIVE_ROOT / filename
    if direct.is_file():
        return direct
    matches = list(Path('/content/drive/MyDrive').rglob(filename))
    return matches[0] if matches else None


WORKDIR.mkdir(parents=True, exist_ok=True)
if not TRAIN_CSV.is_file() or not TRAIN_IMAGE_DIR.is_dir():
    train_bundle = find_drive_file('snu_colab_full_train_bundle.tar')
    if train_bundle is None:
        raise FileNotFoundError('snu_colab_full_train_bundle.tar was not found in MyDrive')
    print(f'Extracting train bundle: {train_bundle}')
    subprocess.run(['tar', '-xf', str(train_bundle), '-C', str(WORKDIR)], check=True)

if not CODE_BUNDLE_PATH.is_file():
    raise FileNotFoundError(f'Upload the code bundle to this exact path: {CODE_BUNDLE_PATH}')
bundle_sha256 = hashlib.sha256(CODE_BUNDLE_PATH.read_bytes()).hexdigest().upper()
if bundle_sha256 != EXPECTED_BUNDLE_SHA256:
    raise RuntimeError(f'Bundle checksum mismatch: {bundle_sha256} != {EXPECTED_BUNDLE_SHA256}')
subprocess.run(['tar', '-xf', str(CODE_BUNDLE_PATH), '-C', str(WORKDIR)], check=True)

for required in (TRAIN_CSV, HOLDOUT_IDS, SEED_STATE):
    if not required.is_file():
        raise FileNotFoundError(required)
if not (ADAPTER_DIR / 'adapter_config.json').is_file():
    raise FileNotFoundError(f'LoRA adapter not found on Drive: {ADAPTER_DIR}')
if not (ADAPTER_DIR / 'adapter_model.safetensors').is_file():
    raise FileNotFoundError(f'adapter_model.safetensors missing in {ADAPTER_DIR}')
OUT_FT.mkdir(parents=True, exist_ok=True)
OUT_V5.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(WORKDIR))
print('Setup complete. Code bundle sha256:', bundle_sha256)


Mounted at /content/drive
Extracting train bundle: /content/drive/MyDrive/snu ai/snu_colab_full_train_bundle.tar
Setup complete. Code bundle sha256: D73299C7C952020D21CFAC0FA64D0E74946B1FF1825340B6873384C8811268B3


## 2. Stage B — LoRA inference on the 1,915-sample fold-0 holdout

Loads the pinned base Qwen3-VL-8B + the `qwen3vl_final` PEFT adapter (the
LoRA notebook's own evaluation entry point) and runs the LoRA-convention
batch loop (no pixel caps, default vision preprocessing) with OOM fallback,
appending per-sample records to a Drive-side `ft_state.jsonl` so an
interrupted run resumes for free.

In [3]:
import csv
import gc
import json
import time

import torch

with HOLDOUT_IDS.open('r', encoding='utf-8-sig', newline='') as file:
    holdout_ids = [row['Id'] for row in csv.DictReader(file)]
assert len(holdout_ids) == HOLDOUT_EXPECTED, len(holdout_ids)

with TRAIN_CSV.open('r', encoding='utf-8-sig', newline='') as file:
    train_rows = list(csv.DictReader(file))
wanted = set(holdout_ids)
va_rows = [row for row in train_rows if row['Id'].strip() in wanted]
assert len(va_rows) == HOLDOUT_EXPECTED, len(va_rows)

gpu_name = torch.cuda.get_device_name(0)
total_vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {gpu_name} ({total_vram_gib:.1f} GiB)')

from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel

load_started = time.perf_counter()
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID, revision=MODEL_REVISION)
processor.tokenizer.padding_side = 'left'
base_model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_ID,
    revision=MODEL_REVISION,
    dtype=torch.bfloat16,
    device_map='auto',
)
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
model.eval()
ft_load_seconds = time.perf_counter() - load_started
lora_b_mean = None
for name, param in model.named_parameters():
    if 'lora_B' in name:
        lora_b_mean = param.abs().mean().item()
        break
print(f'LoRA model loaded in {ft_load_seconds:.1f}s; lora_B mean={lora_b_mean}')
assert lora_b_mean and lora_b_mean > 0, 'adapter weights look untrained or empty'


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition (95.0 GiB)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

LoRA model loaded in 77.8s; lora_B mean=0.008305066265165806


In [4]:
from snu_ordering.hybrid_inference import run_ft_inference

ft_state_path = OUT_FT / 'ft_state.jsonl'
ft_started = time.perf_counter()
records = run_ft_inference(
    processor, model, va_rows, str(TRAIN_IMAGE_DIR),
    batch_size=FT_BATCH_SIZE, state_path=ft_state_path, variant='lora',
)
ft_wall_seconds = time.perf_counter() - ft_started
assert len(records) == HOLDOUT_EXPECTED, len(records)

import ast

answers = {row['Id']: ast.literal_eval(row['Answer']) for row in va_rows}
no_ordering = {row['Id']: row['No_ordering'].strip().lower() == 'true' for row in va_rows}
exact = [record['pred'] == answers[record['Id']] for record in records]
shuffled = [
    record['pred'] == answers[record['Id']]
    for record in records
    if not no_ordering[record['Id']]
]
parse_failures = sum(not record['parse_ok'] for record in records)
em_overall = sum(exact) / len(exact)
em_shuffled = sum(shuffled) / len(shuffled)
print(f'[LoRA fold-0 holdout n={len(records)}] EM overall={em_overall:.4f} (first clean measurement; no recorded baseline)')
print(f'  EM shuffled (No_ordering=False, n={len(shuffled)}): {em_shuffled:.4f}')
print(f'  parse failures: {parse_failures}')

gen_times = sorted(record['gen_time_ms'] for record in records)
timing = {
    'mean_ms': sum(gen_times) / len(gen_times),
    'median_ms': gen_times[len(gen_times) // 2],
    'p90_ms': gen_times[int(len(gen_times) * 0.9)],
    'max_ms': gen_times[-1],
}

with (OUT_FT / 'ft_val_preds.csv').open('w', encoding='utf-8', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=['Id', 'pred', 'parse_ok', 'gen_time_ms'])
    writer.writeheader()
    for record in records:
        writer.writerow({
            'Id': record['Id'],
            'pred': json.dumps(record['pred']),
            'parse_ok': record['parse_ok'],
            'gen_time_ms': record['gen_time_ms'],
        })
(OUT_FT / 'metadata.json').write_text(json.dumps({
    'gpu': gpu_name,
    'base_model': f'{BASE_MODEL_ID}@{MODEL_REVISION}',
    'adapter_dir': str(ADAPTER_DIR),
    'variant': 'lora',
    'model_load_seconds': round(ft_load_seconds, 1),
    'run_wall_seconds': round(ft_wall_seconds, 1),
    'batch_size': FT_BATCH_SIZE,
    'n': len(records),
    'em_overall': round(em_overall, 4),
    'em_shuffled': round(em_shuffled, 4),
    'parse_failures': parse_failures,
    'gen_time_ms': {key: round(value, 1) for key, value in timing.items()},
}, indent=2) + '\n', encoding='utf-8')
print('Wrote', OUT_FT / 'ft_val_preds.csv')
print('Per-sample gen time:', {key: round(value, 1) for key, value in timing.items()})


ft completed=32/1915 run_processed=32/1915 rate=4.18 samples/s eta=7.5 min
ft completed=56/1915 run_processed=56/1915 rate=4.71 samples/s eta=6.6 min
ft completed=80/1915 run_processed=80/1915 rate=4.88 samples/s eta=6.3 min
ft completed=104/1915 run_processed=104/1915 rate=5.02 samples/s eta=6.0 min
ft completed=128/1915 run_processed=128/1915 rate=5.13 samples/s eta=5.8 min
ft completed=152/1915 run_processed=152/1915 rate=5.16 samples/s eta=5.7 min
ft completed=176/1915 run_processed=176/1915 rate=5.22 samples/s eta=5.6 min
ft completed=200/1915 run_processed=200/1915 rate=5.26 samples/s eta=5.4 min
ft completed=232/1915 run_processed=232/1915 rate=5.31 samples/s eta=5.3 min
ft completed=256/1915 run_processed=256/1915 rate=5.34 samples/s eta=5.2 min
ft completed=280/1915 run_processed=280/1915 rate=5.36 samples/s eta=5.1 min
ft completed=304/1915 run_processed=304/1915 rate=5.39 samples/s eta=5.0 min
ft completed=328/1915 run_processed=328/1915 rate=5.41 samples/s eta=4.9 min
ft co

## 3. Release the FT model before Stage C1 (both 8B models must never be co-resident)

In [5]:
del model, base_model, processor
gc.collect()
torch.cuda.empty_cache()
allocated_gib = torch.cuda.memory_allocated() / 1024**3
print(f'CUDA allocated after release: {allocated_gib:.2f} GiB')
assert allocated_gib < 1.0, 'LoRA model still resident; restart the runtime, rerun setup, then continue from Stage C1'


CUDA allocated after release: 0.01 GiB


## 4. Stage C1 — caption-only v5 decomposition for all 1,915 holdout Ids

Runs as a subprocess so CUDA memory is fully returned when it exits. The 193
holdout Ids already covered by the 900-sample run are imported from its
`state.jsonl` (`reused: true`), never regenerated; ~1,722 are new.

In [6]:
from collections import deque
from datetime import datetime, timezone

run_command = [
    sys.executable, '-m', 'preprocessing.qwen3vl_event_chunks_text_ids',
    '--train-csv', str(TRAIN_CSV),
    '--ids-file', str(HOLDOUT_IDS),
    '--output-dir', str(OUT_V5),
    '--seed-state', str(SEED_STATE),
    '--model-id', MODEL_ID,
    '--revision', MODEL_REVISION,
    '--quantization', V5_QUANTIZATION,
    '--batch-size', str(V5_BATCH_SIZE),
    '--max-new-tokens', str(V5_MAX_NEW_TOKENS),
    '--max-retries', str(V5_MAX_RETRIES),
]

state_path = OUT_V5 / 'state.jsonl'
errors_path = OUT_V5 / 'errors.jsonl'
fatal_error_path = OUT_V5 / 'fatal_error.log'
log_path = OUT_V5 / 'qwen_run.log'


def nonempty_line_count(path):
    if not path.is_file():
        return 0
    with path.open('r', encoding='utf-8') as file:
        return sum(bool(line.strip()) for line in file)


print('resuming valid state rows:', nonempty_line_count(state_path))
print('previous error records:', nonempty_line_count(errors_path))
print('full streaming log:', log_path)

tail = deque(maxlen=120)
with log_path.open('a', encoding='utf-8', buffering=1) as log_file:
    marker = f"\n===== run {datetime.now(timezone.utc).isoformat()} =====\n"
    log_file.write(marker)
    print(marker, end='')
    process = subprocess.Popen(
        run_command,
        cwd=WORKDIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        errors='replace',
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        tail.append(line)
    return_code = process.wait()

if return_code != 0:
    print('\n===== INNER ERROR TAIL =====')
    print(''.join(tail), end='')
    raise RuntimeError(
        f'v5 holdout event generation failed with exit code {return_code}. '
        f'The traceback is printed above and saved at {log_path} and {fatal_error_path}'
    )
print('v5 holdout event generation completed successfully.')


resuming valid state rows: 0
previous error records: 0
full streaming log: /content/drive/MyDrive/snu ai/preprocessing/qwen3vl_event_chunks_v5_text_holdout/qwen_run.log

===== run 2026-07-21T05:47:57.186371+00:00 =====
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Reused 193 completed rows from seed state
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition (95.0 GiB), dtype=bfloat16, quantization=none, batch_size=8

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 6364.65it/s]

Loading weights: 100%|██████████| 750/750 [00:01<00:00, 613.64it/s] 
completed=201/1915 run_processed=8/1722 rate=1.87 captions/s eta=15.3 min
complete

## 5. Validate both stage outputs

In [7]:
from collections import Counter

events_path = OUT_V5 / 'events.csv'
v5_metadata = json.loads((OUT_V5 / 'metadata.json').read_text(encoding='utf-8'))
assert v5_metadata['status'] == 'complete', v5_metadata['status']
assert v5_metadata['completed_rows'] == HOLDOUT_EXPECTED, v5_metadata['completed_rows']

with events_path.open('r', encoding='utf-8', newline='') as file:
    reader = csv.DictReader(file)
    event_rows = list(reader)
    fieldnames = reader.fieldnames
assert fieldnames == ['Id', 'Sentence', 'Events', 'Camera', 'Progression'], fieldnames
assert len(event_rows) == HOLDOUT_EXPECTED, len(event_rows)
assert {row['Id'] for row in event_rows} == set(holdout_ids)

state_records = {}
with state_path.open('r', encoding='utf-8') as file:
    for line in file:
        if line.strip():
            payload = json.loads(line)
            state_records[payload['Id']] = payload
reused_rows = sum(1 for payload in state_records.values() if payload.get('reused'))
fallback_ids = sorted(
    sample_id for sample_id, payload in state_records.items() if payload.get('fallback')
)
assert reused_rows == V5_SEED_OVERLAP_EXPECTED, reused_rows

k_distribution = Counter(len(json.loads(row['Events'])) for row in event_rows)
new_times = [
    payload['gen_time_ms']
    for payload in state_records.values()
    if not payload.get('reused') and 'gen_time_ms' in payload
]
print('validated rows:', len(event_rows))
print('K distribution:', dict(sorted(k_distribution.items())))
print('reused from 900-run state:', reused_rows)
print('fallback ids:', fallback_ids)
if new_times:
    new_times.sort()
    print(
        f'new-row gen time ms: mean={sum(new_times) / len(new_times):.0f} '
        f'median={new_times[len(new_times) // 2]:.0f} max={new_times[-1]:.0f}'
    )
ft_metadata = json.loads((OUT_FT / 'metadata.json').read_text(encoding='utf-8'))
print('Stage B recap:', ft_metadata)


validated rows: 1915
K distribution: {1: 387, 2: 892, 3: 452, 4: 184}
reused from 900-run state: 193
fallback ids: ['JqutMU', 'TKOisr']
new-row gen time ms: mean=407 median=405 max=1972
Stage B recap: {'gpu': 'NVIDIA RTX PRO 6000 Blackwell Server Edition', 'base_model': 'Qwen/Qwen3-VL-8B-Instruct@0c351dd01ed87e9c1b53cbc748cba10e6187ff3b', 'adapter_dir': '/content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl_final', 'variant': 'lora', 'model_load_seconds': 77.8, 'run_wall_seconds': 342.2, 'batch_size': 8, 'n': 1915, 'em_overall': 0.53, 'em_shuffled': 0.4882, 'parse_failures': 0, 'gen_time_ms': {'mean_ms': 177.5, 'median_ms': 176.7, 'p90_ms': 180.0, 'max_ms': 426.5}}


## Output

Copy these back into the repo:

- `snu ai/outputs/lora_holdout_infer/{ft_val_preds.csv, ft_state.jsonl, metadata.json}`
  → `outputs/lora_holdout_infer/`
- `snu ai/preprocessing/qwen3vl_event_chunks_v5_text_holdout/{events.csv, state.jsonl, errors.jsonl, metadata.json, qwen_run.log}`
  → `preprocessing/generated/qwen3vl_event_chunks_v5_text_holdout/`

Then run `scripts/run_hybrid_judgment.py` locally for the judgment table.